# YOLO11s-seg solar filaments: controlled architecture comparison

The preprocessing version scores **0.32 after five epochs**, with no reported improvement at 50 epochs (user-reported). The revised YOLO26s-seg run also remains at 0.32 (user-reported). This experiment uses **YOLO11s-seg with bilateral preprocessing**, while preserving **full 2048 × 2048 disks, all-annotator union targets, the grouped train/calibration/report split, D4 averaging, postprocessing rules, and CSV format**. Five epochs remain the initial comparison budget. These observations describe the tested YOLO26 setup; they do not establish a maximum score for YOLO26 or a guaranteed improvement from YOLO11.

YOLO11s-seg is a comparable-size alternative with a different segmentation head and training loss from YOLO26s-seg. Both are supported by the same pinned Ultralytics package. Use `YOLO_MODEL_NAME="yolo26s-seg.pt"` to repeat the baseline with the same settings. Checkpoint selection now compares **best/last only** by default, as requested; optional per-epoch selection remains available.

Methods adapted from the supplied notebooks:

- **Panoptic morphology:** radial background flattening, relative absorption, multi-scale black-hat salience and small-gap bridging. Salience guides contrast and a calibration candidate; it never creates unrestricted morphology-only predictions or training labels.
- **U-Net OOF:** per-image solar-disk mean/SD normalization, eight invertible rotation/flip views, averaged mask probabilities, and connected-component low/high/mean/area filters. YOLO probabilities have different calibration, so the U-Net's exact keep rule competes with looser rules rather than becoming a fixed assumption. The ten U-Net weights and architecture are not used.
- **Original YOLO:** single-view binary-mask union → disjoint connected components remains a calibration candidate. Both alternatives use the same trained checkpoint and input transform.

Two GPUs train with DDP and **two data-loader workers per GPU, four total**. YOLO's extra validation-loader workers are disabled. Calibration, report and test inference use one independent model replica per GPU, fed by a bounded **four-worker CPU pool**, with one native image/view per GPU call. A one-GPU or CPU fallback is supported and printed explicitly. This notebook is self-contained. Run all cells on Kaggle with **2 × T4**; upload only `solar_filament_yolo11s_submission.csv`.


## Quick repair of an existing CSV (no retraining or model inference)

If the previous submission was rejected for overlapping masks, run **only the next cell** after setting `RUN_REPAIR_EXISTING_CSV=True` and `REPAIR_INPUT_CSV` to that CSV. It imports its own libraries and does not need any trained model, dataset paths or other notebook cells. The source CSV is preserved; upload the new `*_nonoverlap.csv` file.

For each image, this applies the original union -> 8-connected components operation, re-encodes each component using the original COCO counts format, and numbers the rows again. The default repair threshold is 1 pixel to preserve the input foreground; set `min_area=20` in the call to reproduce the original notebook's component filter too. A disk-backed spool handles interleaved image rows without storing all masks in RAM. The repair audits every written RLE for overlap and writes a separate JSON report.

After a fresh run of the full notebook, use its normal submission; the final pipeline now enforces this same non-overlap rule. Old scores selected using the invalid instance mode should be recalibrated; no new score is claimed by repairing the CSV.


In [ ]:
# Standalone recovery: this cell does not train a model or run GPU inference.
import csv
import json
import os
import sqlite3
import tempfile
from contextlib import closing
from pathlib import Path
import cv2
import numpy as np
from pycocotools import mask as mask_utils


def repair_submission_overlaps(input_path, output_path=None, min_area=1):
    input_path = Path(input_path)
    output_path = (
        Path(output_path)
        if output_path is not None
        else input_path.with_name(input_path.stem + "_nonoverlap.csv")
    )
    if input_path.resolve() == output_path.resolve():
        raise ValueError(
            "Choose a separate output file; the rejected CSV is preserved."
        )
    if min_area < 1:
        raise ValueError("min_area must be at least 1 pixel.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    csv.field_size_limit(16 * 1024 * 1024)

    def encode_counts(binary_mask):
        # Exact encoding operation from the original 0.36 notebook.
        encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
        return encoded["counts"].decode("utf-8")

    report = dict(
        input_csv=str(input_path),
        output_csv=str(output_path),
        input_rows=0,
        output_rows=0,
        images_processed=0,
        images_with_input_overlaps=0,
        input_overlap_assignments=0,
        min_area=int(min_area),
        non_overlapping_masks=False,
    )
    with tempfile.TemporaryDirectory(
        prefix="filament_nonoverlap_", dir=output_path.parent
    ) as directory:
        temporary_csv = Path(directory) / "submission.csv"
        with closing(
            sqlite3.connect(str(Path(directory) / "input.sqlite"))
        ) as connection:
            connection.execute(
                "CREATE TABLE masks (filament_id TEXT PRIMARY KEY, stem TEXT, counts TEXT)"
            )
            with input_path.open(newline="", encoding="utf-8") as stream:
                reader = csv.DictReader(stream)
                if reader.fieldnames != ["filament_id", "segmentation_rle"]:
                    raise ValueError(
                        "Expected exactly filament_id,segmentation_rle columns."
                    )
                for row in reader:
                    if (
                        set(row) != {"filament_id", "segmentation_rle"}
                        or None in row.values()
                    ):
                        raise ValueError("Malformed submission row.")
                    fid, counts = row["filament_id"], row["segmentation_rle"]
                    if not fid or not counts or "_" not in fid:
                        raise ValueError("Empty or invalid filament ID / RLE counts.")
                    stem, suffix = fid.rsplit("_", 1)
                    if not stem or not suffix:
                        raise ValueError(f"Invalid filament ID: {fid}")
                    try:
                        connection.execute(
                            "INSERT INTO masks VALUES (?, ?, ?)", (fid, stem, counts)
                        )
                    except sqlite3.IntegrityError as error:
                        raise ValueError(f"Duplicate filament ID: {fid}") from error
                    report["input_rows"] += 1
            connection.execute("CREATE INDEX masks_stem ON masks(stem)")
            connection.commit()
            with temporary_csv.open("w", newline="", encoding="utf-8") as stream:
                writer = csv.writer(stream)  # Same default writer as original notebook.
                writer.writerow(["filament_id", "segmentation_rle"])
                for (stem,) in connection.execute(
                    "SELECT DISTINCT stem FROM masks ORDER BY stem"
                ):
                    union = np.zeros((2048, 2048), np.uint8)
                    overlap_assignments = 0
                    for (counts,) in connection.execute(
                        "SELECT counts FROM masks WHERE stem=?", (stem,)
                    ):
                        rle = {"size": [2048, 2048], "counts": counts.encode("utf-8")}
                        mask = mask_utils.decode(rle).astype(np.uint8)
                        if mask.shape != (2048, 2048) or not mask.any():
                            raise ValueError(f"Empty or invalid mask for {stem}")
                        overlap_assignments += int(np.count_nonzero(union & mask))
                        union |= mask
                    number, labels, stats, _ = cv2.connectedComponentsWithStats(
                        union, connectivity=8
                    )
                    keep = [
                        i
                        for i in range(1, number)
                        if stats[i, cv2.CC_STAT_AREA] >= min_area
                    ]
                    keep.sort(
                        key=lambda i: (
                            -int(stats[i, cv2.CC_STAT_AREA]),
                            int(stats[i, cv2.CC_STAT_TOP]),
                            int(stats[i, cv2.CC_STAT_LEFT]),
                        )
                    )
                    for counter, label in enumerate(keep, start=1):
                        writer.writerow(
                            [f"{stem}_{counter}", encode_counts(labels == label)]
                        )
                    report["images_processed"] += 1
                    report["output_rows"] += len(keep)
                    report["images_with_input_overlaps"] += int(overlap_assignments > 0)
                    report["input_overlap_assignments"] += overlap_assignments

        # Independently verify the actual written RLEs before publishing the CSV.
        occupied = {}
        seen = set()
        with temporary_csv.open(newline="", encoding="utf-8") as stream:
            for row in csv.DictReader(stream):
                fid = row["filament_id"]
                if fid in seen:
                    raise ValueError(f"Duplicate output ID: {fid}")
                seen.add(fid)
                stem = fid.rsplit("_", 1)[0]
                rle = {
                    "size": [2048, 2048],
                    "counts": row["segmentation_rle"].encode("utf-8"),
                }
                mask = mask_utils.decode(rle)
                if (
                    mask.shape != (2048, 2048)
                    or not mask.any()
                    or encode_counts(mask) != row["segmentation_rle"]
                ):
                    raise ValueError(f"Invalid repaired mask: {fid}")
                if stem in occupied:
                    overlap = mask_utils.merge([occupied[stem], rle], intersect=True)
                    if int(mask_utils.area(overlap)):
                        raise ValueError(f"Repaired masks still overlap: {fid}")
                    occupied[stem] = mask_utils.merge([occupied[stem], rle])
                else:
                    occupied[stem] = rle
        if len(seen) != report["output_rows"]:
            raise ValueError("Repaired CSV row-count mismatch.")
        os.replace(temporary_csv, output_path)
    report["non_overlapping_masks"] = True
    report["output_overlap_pixels"] = 0
    output_path.with_suffix(".audit.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    return output_path, report


RUN_REPAIR_EXISTING_CSV = False
REPAIR_INPUT_CSV = Path(
    "solar_filament_yolo11s_submission.csv"
)  # Or Path("submission.csv").
if RUN_REPAIR_EXISTING_CSV:
    repaired_csv_path, repair_report = repair_submission_overlaps(REPAIR_INPUT_CSV)
    print(json.dumps(repair_report, indent=2))
    print(f"Upload this corrected CSV: {repaired_csv_path}")

## Why change the approach?

More epochs alone did not improve the reported result. That observation does not identify the cause. This experiment tests contrast consistency, prediction stability and component selection while retaining the YOLO model and targets.

| Supplied reference                     | Adaptation here                                                                          | Limit                                                                                                                                    |
| -------------------------------------- | ---------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------- |
| Panoptic morphology, Stage 02          | Bounded radial correction; multi-scale absorption/black-hat salience; restricted closing | No hard 0.945-radius crop, which could remove limb filaments. Failed disk fits preserve the raw image.                                   |
| U-Net, Test images                     | Normalize using this image's in-disk mean and SD                                         | Map the normalized values to uint8 before YOLO's usual `/255`; keep three identical channels. Retrain for this transform.                |
| U-Net, Inference                       | D4 rotation/flip averaging, inverse-transform every map                                  | Decode actual YOLO mask logits before their normal binary threshold. Detection confidence and pixel probability are separate quantities. |
| U-Net, From probabilities to filaments | Low threshold defines components; peak, mean and area decide whether to keep them        | Tune on calibration only. U-Net thresholds are not assumed to transfer.                                                                  |

The pretrained model, polygon target conversion and CSV encoder remain the same. No U-Net ensemble, new model, test annotations, pseudo-labels or external MAGFiLO labels are introduced. The source notebooks' prose and code are references, not instructions to execute their workflows.

Local PQ retains the existing evaluator: IoU **> 0.5**, pooled over annotator-image records, including its all-hit handling of overlapping ground truth. Calibration selects the checkpoint and extraction rule; the separate report partition is evaluated only after selection. Neither local score is a measured public score.

The soft decoder mirrors the pinned [Ultralytics native mask decoding](https://github.com/ultralytics/ultralytics/blob/v8.4.171/ultralytics/utils/ops.py) through logit upsampling and box cropping, retaining sigmoid values for averaging. See also the pinned [segmentation predictor](https://github.com/ultralytics/ultralytics/blob/v8.4.171/ultralytics/models/yolo/segment/predict.py) and [data-loader implementation](https://github.com/ultralytics/ultralytics/blob/v8.4.171/ultralytics/models/yolo/detect/train.py).


## 1. Dependencies

Keep Ultralytics pinned to **8.4.171**, the version used for the preceding YOLO26 experiment. It supports both YOLO11s-seg and YOLO26s-seg, so changing model family does not also change the package version. Keep Kaggle's existing CUDA-enabled Torch. Resolved package versions and the complete experiment configuration are saved with the run.


In [ ]:
%pip -q install --no-cache-dir ultralytics==8.4.171 pycocotools scikit-learn scipy tqdm threadpoolctl psutil


In [ ]:
from pathlib import Path
import csv, gc, hashlib, json, os, queue, random, re, shutil, time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version

for _thread_key in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[_thread_key] = "1"  # Also inherited by Ultralytics DDP subprocesses.
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import ultralytics
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from sklearn.model_selection import GroupShuffleSplit
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm
from ultralytics import YOLO
from ultralytics.engine.results import Results
from ultralytics.models.yolo.segment import SegmentationPredictor, SegmentationTrainer
from ultralytics.utils import ops

assert (
    ultralytics.__version__ == "8.4.171"
), "Restart the kernel after installing the pinned version."
csv.field_size_limit(16 * 1024 * 1024)
cv2.setNumThreads(1)
torch.set_num_threads(1)
CPU_THREAD_LIMIT = threadpool_limits(limits=1)

## 2. Configuration

The selected input transform is `bilateral`: OpenCV bilateral filtering reduces noise while preserving intensity edges. Its starting settings are `bilateral_d=9`, `bilateral_sigma_color=25.0` (uint8 intensity units), and `bilateral_sigma_space=5.0` (native pixels), all editable in `PREPROCESSING_CONFIG`. Train, calibration, report and test use exactly the same deterministic transform once. `solar` and `raw` remain available for comparison with the same split and targets. Changing the transform or its parameters requires retraining; these settings are included in the run fingerprint so old datasets, checkpoints and prediction caches cannot be silently reused.

The default model is now **`yolo11s-seg.pt`**. The small scale keeps capacity in the same range as the previous YOLO26s-seg run, making model-family comparison more useful than changing architecture and model size together. Keep the split and training budget fixed when comparing bilateral filtering with the previous `solar` transform. Five epochs may not be enough for every architecture to converge; inspect validation curves before choosing a longer follow-up run. No filament-PQ ranking is implied by COCO benchmark results.

The initial budget is **5 epochs**, with a shorter warmup and nominal batch 16 (gradient accumulation with a global batch of two). **Only `best.pt` and `last.pt` compete on calibration PQ by default** (`SAVE_EVERY_EPOCH=False`). Set this flag to `True` only when you want optional per-epoch PQ selection. Identical checkpoint files are deduplicated. More epochs are an optional experiment, not the proposed fix.

Eight D4 views are averaged with equal weight, including zero maps when a view misses a detection. The single-view binary baseline is cached separately. Calibration compares it with probability hysteresis and a small number of morphology-guided rules. Calibration postprocessing can be changed using retained maps without another forward pass; extending the confidence grid or changing TTA creates a new fingerprint/cache.

Dates remain grouped across approximately **80% train / 10% calibration / 10% report**. All annotators of one physical image stay together. Report metrics must not be used for selection. For a clean raw-input comparison set only `PREPROCESSING_METHOD="raw"` before Run All.

**Disk use:** `PREDICTION_CACHE_POLICY="bounded"` keeps at most the incumbent and current checkpoint's calibration maps while selecting. Losing-checkpoint maps are removed after scoring. Report/test maps are processed incrementally and retained only for the preview images; metrics, CSV masks and all weight files are preserved. The preflight budgets this peak, not the sum of every checkpoint and image. YOLO's optional RGB `.npy` image cache is disabled if it would exceed available space. `resource_plan.json` prints the breakdown. Set the policy to `"all"` only with enough disk to retain every probability map.


In [ ]:
SEED = 42
DATA_ROOT_OVERRIDE = ""  # Optional explicit competition root.
OUTPUT_ROOT = Path.cwd()
PREPROCESSING_METHOD = "bilateral"  # bilateral, solar, raw, disk_raw, radial, clahe, percentile, gamma
PREPROCESSING_CONFIG = {
    "bilateral_d": 9,  # Neighborhood diameter in native pixels.
    "bilateral_sigma_color": 25.0,  # Intensity smoothing in the uint8 [0, 255] range.
    "bilateral_sigma_space": 5.0,  # Spatial smoothing in native pixels.
    "clahe_clip": 1.2,
    "clahe_grid": [16, 16],
    "clahe_strength": 0.25,
    "radial_strength": 0.65,
    "radial_gain_limits": [0.80, 1.25],
    "percentiles": [0.5, 99.5],
    "gamma": 0.9,
    "disk_margin": 1.02,
    "disk_stats_radius": 0.95,
    "normalized_mean": 128.0,
    "normalized_std": 32.0,
    "morphology_strength": 0.25,
    "salience_size": 1024,
    "blackhat_kernels": [9, 19, 33],
}
SPLIT_GROUP = "date"
VALIDATION_FRACTION = 0.20
CALIBRATION_SHARE = 0.50
IMAGE_SIZE = 2048
YOLO_MODEL_NAME = "yolo11s-seg.pt"  # Previous model: "yolo26s-seg.pt".
MODEL_TAG = Path(YOLO_MODEL_NAME).stem.removesuffix("-seg")
NUM_EPOCHS = 100
PATIENCE = 5
TARGET_MIN_AREA = 20  # Same union-component threshold as the supplied notebook.
NUM_WORKERS = min(4, os.cpu_count() or 1)
GPU_IDS = list(range(min(2, torch.cuda.device_count())))
TRAIN_DEVICE = GPU_IDS if len(GPU_IDS) > 1 else (GPU_IDS[0] if GPU_IDS else "cpu")
INFERENCE_DEVICES = [f"cuda:{i}" for i in GPU_IDS] or ["cpu"]
BATCH_SIZE = max(1, len(GPU_IDS))
YOLO_WORKERS = NUM_WORKERS // max(1, len(GPU_IDS)) if GPU_IDS else 0
assert YOLO_WORKERS * max(1, len(GPU_IDS)) <= 4
MAX_DETECTIONS = 300 # Previous: 100
RAM_RESERVE_GIB = 6.0
DISK_RESERVE_GIB = 6.0
DISK_LIMIT_GIB = 57.6
PREDICTION_CACHE_POLICY = (
    "bounded"  # bounded: winner + previews; all: retain every map.
)
RUN_TRAINING = True
EXISTING_CHECKPOINT = (
    ""  # Only used if RUN_TRAINING=False; matching run_config.json required.
)
CREATE_SUBMISSION = True
REBUILD_YOLO_DATASET = (
    False  # Fingerprints prevent reusing another experiment's dataset.
)
PREDICTION_PREVIEW_COUNT = 10
PREPROCESS_PREVIEW_COUNT = 3
CONFIDENCE_CANDIDATES = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
AREA_CANDIDATES = [0, 20, 80, 200]
TTA_VIEWS = 8  # 1, 2, 4 or 8; exact 90-degree rotations/reflections, no interpolation.
MASK_DECODE_CHUNK = 4  # At most four full-size float instance masks per GPU.
KEEP_RULES = [
    dict(t_lo=0.35, t_hi=0.70, min_meanp=0.50, min_area=80),
    dict(t_lo=0.35, t_hi=0.85, min_meanp=0.65, min_area=200),
    dict(t_lo=0.50, t_hi=0.90, min_meanp=0.75, min_area=200),
    dict(t_lo=0.35, t_hi=0.99, min_meanp=0.85, min_area=200),  # U-Net reference rule.
]
MORPHOLOGY_RULES = [
    dict(t_lo=0.35, t_hi=0.75, min_meanp=0.55, min_area=165, bridge_px=3),
    dict(t_lo=0.35, t_hi=0.85, min_meanp=0.65, min_area=200, bridge_px=5),
]
SAVE_EVERY_EPOCH = False
CLIP_TO_DISK_CANDIDATES = [False, True]
PREDICT_NMS_IOU = (
    0.70  # Fixed; not described as an optimal or instance-merging threshold.
)
PQ_MATCH_IOU = 0.50
GIB = 1024**3

TRAIN_OPTIONS = dict(
    epochs=NUM_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=TRAIN_DEVICE,
    workers=YOLO_WORKERS,
    patience=PATIENCE,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=0.5,
    nbs=16,
    save_period=1 if SAVE_EVERY_EPOCH else -1,
    amp=bool(GPU_IDS),
    seed=SEED,
    deterministic=True,
    max_det=MAX_DETECTIONS,
    fliplr=0.5,
    flipud=0.5,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.10,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=0,
    overlap_mask=True,
    mask_ratio=4,
    augmentations=[],
    plots=True,
    verbose=True,
)
assert PREDICTION_CACHE_POLICY in {"bounded", "all"}
assert TTA_VIEWS in {1, 2, 4, 8}
assert PREPROCESSING_METHOD in {
    "bilateral",
    "solar",
    "raw",
    "disk_raw",
    "radial",
    "clahe",
    "percentile",
    "gamma",
}
assert SPLIT_GROUP in {"date", "filename"}
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if GPU_IDS:
    torch.cuda.manual_seed_all(SEED)
print("Pretrained model:", YOLO_MODEL_NAME)
print(
    "Train devices:",
    TRAIN_DEVICE,
    "| batch:",
    BATCH_SIZE,
    "| inference:",
    INFERENCE_DEVICES,
)
if not GPU_IDS and RUN_TRAINING:
    print("CPU fallback detected. Full-resolution training will be slow.")

print(
    f"CPU workers: {NUM_WORKERS}; DDP loaders: {YOLO_WORKERS} per rank; validation loaders: 0; TTA: {TTA_VIEWS}"
)

In [ ]:
def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=True), encoding="utf-8")
    os.replace(tmp, path)


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def find_dataset_root():
    expected = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    roots = (
        [Path(DATA_ROOT_OVERRIDE)]
        if DATA_ROOT_OVERRIDE
        else [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]
    )
    for parent in roots:
        if not parent.is_dir():
            continue
        for annotation in parent.glob("**/" + expected):
            root = annotation.parent.parent
            if (root / "train/train_images").is_dir() and (
                root / "test/test_images"
            ).is_dir():
                return root.resolve()
    raise FileNotFoundError("Attach the competition dataset or set DATA_ROOT_OVERRIDE.")


def read_gray(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise OSError(f"Unreadable image: {path}")
    if image.shape != (IMAGE_SIZE, IMAGE_SIZE):
        raise ValueError(
            f"Expected native {IMAGE_SIZE} square image, got {image.shape}: {path}"
        )
    return image


def bounded_map(function, items, workers=NUM_WORKERS):
    iterator = iter(items)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        pending = deque()
        for _ in range(workers):
            item = next(iterator, None)
            if item is not None:
                pending.append(pool.submit(function, item))
        while pending:
            yield pending.popleft().result()  # Propagate every worker failure.
            item = next(iterator, None)
            if item is not None:
                pending.append(pool.submit(function, item))


DATA_ROOT = find_dataset_root()
TRAIN_IMAGE_DIR = DATA_ROOT / "train/train_images"
TEST_IMAGE_DIR = DATA_ROOT / "test/test_images"
ANNOTATION_PATH = DATA_ROOT / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
coco = COCO(str(ANNOTATION_PATH))
records = pd.DataFrame(
    [
        dict(
            image_id=i,
            file_name=m["file_name"],
            date=str(m.get("date_captured", ""))[:10],
        )
        for i, m in coco.imgs.items()
    ]
)
if SPLIT_GROUP == "date":
    records["group"] = records["file_name"].map(
        lambda n: re.match(r"^(\d{8})", Path(n).name).group(1)
    )
else:
    records["group"] = records["file_name"]
splitter = GroupShuffleSplit(
    n_splits=1, test_size=VALIDATION_FRACTION, random_state=SEED
)
train_idx, holdout_idx = next(splitter.split(records, groups=records["group"]))
train_records = records.iloc[train_idx].reset_index(drop=True)
holdout = records.iloc[holdout_idx].reset_index(drop=True)
splitter2 = GroupShuffleSplit(
    n_splits=1, train_size=CALIBRATION_SHARE, random_state=SEED + 1
)
cal_idx, report_idx = next(splitter2.split(holdout, groups=holdout["group"]))
calibration_records = holdout.iloc[cal_idx].reset_index(drop=True)
report_records = holdout.iloc[report_idx].reset_index(drop=True)
for left, right in [
    (train_records, calibration_records),
    (train_records, report_records),
    (calibration_records, report_records),
]:
    assert set(left.file_name).isdisjoint(right.file_name)
    assert set(left.group).isdisjoint(right.group)


def physical_rows(frame):
    return (
        frame.groupby("file_name", sort=True)["image_id"]
        .agg(list)
        .reset_index(name="image_ids")
        .to_dict("records")
    )


VERSIONS = {
    p: version(p)
    for p in [
        "ultralytics",
        "torch",
        "torchvision",
        "numpy",
        "opencv-python",
        "pycocotools",
        "scikit-learn",
        "scipy",
    ]
}
RUN_CONFIG = dict(
    pipeline="solar-d4-hysteresis-pq-v3",
    seed=SEED,
    image_size=IMAGE_SIZE,
    preprocessing_method=PREPROCESSING_METHOD,
    preprocessing=PREPROCESSING_CONFIG,
    target="all_annotators_union_connected_components",
    target_min_area=TARGET_MIN_AREA,
    split_group=SPLIT_GROUP,
    annotation_sha256=file_sha256(ANNOTATION_PATH),
    train=sorted(train_records.file_name.unique()),
    calibration=sorted(calibration_records.file_name.unique()),
    report=sorted(report_records.file_name.unique()),
    model=YOLO_MODEL_NAME,
    training=TRAIN_OPTIONS,
    versions=VERSIONS,
    resources=dict(
        cpu_workers=NUM_WORKERS,
        train_workers_per_rank=YOLO_WORKERS,
        validation_workers=0,
        inference_devices=INFERENCE_DEVICES,
    ),
    trainer="four_total_train_workers_zero_val_workers_v1",
    selection=dict(
        confidences=CONFIDENCE_CANDIDATES,
        areas=AREA_CANDIDATES,
        tta_views=TTA_VIEWS,
        soft_decoder="native_sigmoid_chunked_v1",
        keep_rules=KEEP_RULES,
        morphology_rules=MORPHOLOGY_RULES,
        disk_clipping=CLIP_TO_DISK_CANDIDATES,
        predict_nms_iou=PREDICT_NMS_IOU,
        pq_match_iou=PQ_MATCH_IOU,
    ),
)


def training_signature(config):
    return {key: value for key, value in config.items() if key != "selection"}


RUN_FINGERPRINT = fingerprint(training_signature(RUN_CONFIG))
EXPERIMENT_DIR = (
    OUTPUT_ROOT
    / f"preprocessing_{PREPROCESSING_METHOD}_union_v3_{RUN_FINGERPRINT[:10]}"
)
YOLO_DATA_DIR = EXPERIMENT_DIR / "yolo_filament_dataset"
RUNS_DIR = EXPERIMENT_DIR / "yolo_runs"
RUN_NAME = f"{MODEL_TAG}_full_resolution"
DATA_YAML = YOLO_DATA_DIR / "filament.yaml"
PREVIEW_DIR = EXPERIMENT_DIR / "previews"
SUBMISSION_PATH = "a_submission.csv"
save_json(EXPERIMENT_DIR / "run_config.json", RUN_CONFIG)
for label, frame in [
    ("train", train_records),
    ("calibration", calibration_records),
    ("report", report_records),
]:
    print(
        label,
        ":",
        frame.file_name.nunique(),
        "physical images,",
        len(frame),
        "annotator records,",
        frame.group.nunique(),
        "groups",
    )
    frame.to_csv(EXPERIMENT_DIR / f"{label}_split.csv", index=False)
print("Dataset:", DATA_ROOT, "\nOutputs:", EXPERIMENT_DIR)

## 3. Bilateral preprocessing and optional solar features

Bilateral filtering is applied directly to the native uint8 grayscale image before model input. It combines spatial and intensity similarity to smooth noise while limiting smoothing across filament edges. The `bilateral` branch runs independently of disk fitting and preserves image shape, dtype and coordinates. It is the selected transform, applied once through the shared `preprocess_grayscale` function for exported training/calibration images and report/test inference. The preprocessing preview compares raw, bilateral, solar and CLAHE inputs.

The following disk normalization and contrast operations are retained for the optional `solar` transform; disk geometry and salience also support existing postprocessing candidates.

Disk geometry is fitted from a small image with center/radius/shape/residual checks. Failed fits preserve the raw input and disable morphology guidance. Annular background correction is bounded. Mean/SD are measured within the fitted disk's inner 95%, then mapped to uint8 for the unchanged YOLO input contract. Only continuous salience features are computed at 1024; images, training coordinates and final masks remain 2048.

Relative absorption and 9/19/33-pixel black-hat filters follow the morphology notebook. Their robust MAD-normalized salience mildly darkens filament candidates in `solar` mode. The outer disk is retained; there is no fixed inner-radius crop. No labels enter preprocessing. Preview input and target alignment before training.


In [ ]:
def estimate_disk(image):
    h, w = image.shape
    small = cv2.resize(image, (256, 256), interpolation=cv2.INTER_AREA)
    blurred = cv2.GaussianBlur(small, (5, 5), 0)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea)
    if len(contour) < 20 or cv2.contourArea(contour) < 0.15 * small.size:
        return None
    (cx, cy), (axis1, axis2), _ = cv2.fitEllipse(contour)
    radius = (axis1 + axis2) / 4
    points = contour[:, 0, :].astype(np.float32)
    residual = np.percentile(
        np.abs(np.linalg.norm(points - [cx, cy], axis=1) - radius), 90
    )
    good = (
        64 < radius < 132
        and abs(cx - 127.5) < 32
        and abs(cy - 127.5) < 32
        and min(axis1, axis2) / max(axis1, axis2) > 0.90
        and residual / radius < 0.06
        and points.min() > 0
        and points.max() < 255
    )
    if not good:
        return None
    return {
        "cx": float(cx * w / 256),
        "cy": float(cy * h / 256),
        "radius": float(radius * min(h, w) / 256),
        "relative_residual": float(residual / radius),
    }


def disk_radius_grid(shape, disk):
    yy, xx = np.ogrid[: shape[0], : shape[1]]
    return (
        np.sqrt((xx - disk["cx"]) ** 2 + (yy - disk["cy"]) ** 2).astype(np.float32)
        / disk["radius"]
    )


def disk_support(shape, disk, margin=None):
    if disk is None:
        return np.ones(shape, dtype=bool)
    return disk_radius_grid(shape, disk) <= (
        PREPROCESSING_CONFIG["disk_margin"] if margin is None else margin
    )


def preprocess_grayscale(image, method=None, config=None):
    method = PREPROCESSING_METHOD if method is None else method
    cfg = PREPROCESSING_CONFIG if config is None else config
    if image.ndim != 2 or image.dtype != np.uint8:
        raise ValueError("Preprocessing expects uint8 grayscale.")
    if method == "solar":
        return solar_preprocess(image, cfg)
    if method == "raw":
        return image.copy()
    if method == "bilateral":
        # Apply once at native resolution; all model-input paths share this function.
        return cv2.bilateralFilter(
            image,
            d=cfg["bilateral_d"],
            sigmaColor=cfg["bilateral_sigma_color"],
            sigmaSpace=cfg["bilateral_sigma_space"],
        )
    disk = estimate_disk(image)
    if disk is None:
        return image.copy()
    rr = disk_radius_grid(image.shape, disk)
    support = rr <= cfg["disk_margin"]
    taper = np.clip((1.02 - rr) / 0.12, 0, 1)
    original = image.astype(np.float32)
    if method == "disk_raw":
        result = np.where(support, image, 0)
    elif method == "clahe":
        enhanced = cv2.createCLAHE(
            clipLimit=cfg["clahe_clip"], tileGridSize=tuple(cfg["clahe_grid"])
        ).apply(image)
        alpha = cfg["clahe_strength"] * taper
        result = original * (1 - alpha) + enhanced.astype(np.float32) * alpha
    elif method == "radial":
        # Estimate broad illumination from annular medians, not dark filament pixels.
        small = cv2.resize(image, (512, 512), interpolation=cv2.INTER_AREA)
        scaled = {
            **disk,
            "cx": disk["cx"] * 512 / image.shape[1],
            "cy": disk["cy"] * 512 / image.shape[0],
            "radius": disk["radius"] * 512 / min(image.shape),
        }
        rsmall = disk_radius_grid(small.shape, scaled)
        centers = (np.arange(64) + 0.5) / 64
        profile = np.full(64, np.nan, dtype=np.float32)
        for i in range(64):
            pixels = small[(rsmall >= i / 64) & (rsmall < (i + 1) / 64)]
            if pixels.size >= 16:
                profile[i] = np.median(pixels)
        valid = np.isfinite(profile) & (profile > 5)
        if valid.sum() < 32:
            return image.copy()
        profile = np.interp(centers, centers[valid], profile[valid]).astype(np.float32)
        profile = cv2.GaussianBlur(profile[None, :], (9, 1), 0).ravel()
        target = float(np.median(profile[centers < 0.65]))
        background = np.interp(rr.ravel(), centers, profile).reshape(image.shape)
        gain = np.clip(target / np.maximum(background, 10), *cfg["radial_gain_limits"])
        result = original * (1 + cfg["radial_strength"] * taper * (gain - 1))
    elif method == "percentile":
        low, high = np.percentile(image[rr < 0.95], cfg["percentiles"])
        enhanced = np.clip((original - low) * 255 / max(float(high - low), 1), 0, 255)
        result = original * (1 - taper) + enhanced * taper
    elif method == "gamma":
        enhanced = 255 * (original / 255) ** cfg["gamma"]
        result = original * (1 - taper) + enhanced * taper
    else:
        raise ValueError(f"Unknown method: {method}")
    return np.ascontiguousarray(np.clip(np.rint(result), 0, 255).astype(np.uint8))


def morphology_salience_u8(image, disk, config=None):
    """Robust absorption/black-hat z score, stored as round(32*z), clipped to [0,255]."""
    cfg = PREPROCESSING_CONFIG if config is None else config
    if disk is None:
        return np.zeros(image.shape, np.uint8)
    size = int(cfg["salience_size"])
    small = cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)
    scaled = {
        **disk,
        "cx": disk["cx"] * size / image.shape[1],
        "cy": disk["cy"] * size / image.shape[0],
        "radius": disk["radius"] * size / min(image.shape),
    }
    inside = disk_radius_grid(small.shape, scaled) <= cfg["disk_stats_radius"]
    if inside.sum() < 100:
        return np.zeros(image.shape, np.uint8)
    gray = small.astype(np.float32)
    scale = size / 1024
    background = 0.65 * cv2.GaussianBlur(
        gray, (0, 0), 28 * scale
    ) + 0.35 * cv2.GaussianBlur(gray, (0, 0), 14 * scale)
    salience = 0.50 * np.clip((background - gray) / np.maximum(background, 25), 0, 1)
    for weight, width in zip([0.15, 0.20, 0.15], cfg["blackhat_kernels"]):
        width = max(3, int(round(width * scale)) | 1)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (width, width))
        salience += (
            weight
            * cv2.morphologyEx(small, cv2.MORPH_BLACKHAT, kernel).astype(np.float32)
            / 255
        )
    values = salience[inside]
    median = float(np.median(values))
    mad = max(1.4826 * float(np.median(np.abs(values - median))), 1e-5)
    z = np.clip((salience - median) / mad, 0, 255 / 32)
    z = cv2.resize(z, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)
    z[~disk_support(image.shape, disk)] = 0
    return np.ascontiguousarray(np.rint(z * 32).astype(np.uint8))


def solar_preprocess(image, cfg):
    disk = estimate_disk(image)
    if disk is None:
        return image.copy()
    corrected = preprocess_grayscale(image, method="radial", config=cfg).astype(
        np.float32
    )
    rr = disk_radius_grid(image.shape, disk)
    values = corrected[rr <= cfg["disk_stats_radius"]]
    mean, std = float(values.mean()), float(values.std())
    if std < 1.0:
        return image.copy()
    normalized = (
        cfg["normalized_mean"] + cfg["normalized_std"] * (corrected - mean) / std
    )
    z = morphology_salience_u8(image, disk, cfg).astype(np.float32) / 32
    normalized -= cfg["morphology_strength"] * 8 * np.minimum(z, 6)
    # Smoothly retain the observed limb/background rather than hard-erasing it.
    blend = np.clip((cfg["disk_margin"] - rr) / 0.04, 0, 1)
    result = normalized * blend + image.astype(np.float32) * (1 - blend)
    return np.ascontiguousarray(np.clip(np.rint(result), 0, 255).astype(np.uint8))

## 4. Union targets and dataset preparation

Every annotator of the same training/calibration filename contributes to its union. Components below 20 native pixels are dropped, as before. Nearby instances that touch after union become one training component; this is an explicit consequence of the requested target policy. Evaluation GT is never unioned or area-filtered. Polygon conversion can fill holes and cannot exactly preserve all raster detail; previews show the actual exported target.

Preparation uses bounded threads (also works in Windows notebooks). Missing/corrupt images raise errors. A manifest prevents stale data reuse. Image copies are lossless grayscale PNG; YOLO reads three identical channels. Only train and calibration images are exported; report images are never provided to the trainer.


In [ ]:
def merged_annotation_mask(coco_api, image_ids):
    first = coco_api.imgs[image_ids[0]]
    shape = (first["height"], first["width"])
    merged = np.zeros(shape, np.uint8)
    for image_id in image_ids:
        metadata = coco_api.imgs[image_id]
        if (
            metadata["file_name"] != first["file_name"]
            or (metadata["height"], metadata["width"]) != shape
        ):
            raise ValueError("Union records must describe the same physical image.")
        for ann in coco_api.loadAnns(coco_api.getAnnIds(imgIds=[image_id])):
            merged |= coco_api.annToMask(ann).astype(np.uint8)
    return merged


def mask_to_yolo_segments(mask, min_area=TARGET_MIN_AREA):
    h, w = mask.shape
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )
    segments = []
    for idx in range(1, count):
        if stats[idx, cv2.CC_STAT_AREA] < min_area:
            continue
        x, y, bw, bh, _ = stats[idx]
        crop = (labels[y : y + bh, x : x + bw] == idx).astype(np.uint8)
        contours, _ = cv2.findContours(crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)
        if len(contour) < 3 or cv2.contourArea(contour) <= 0 or bw < 2 or bh < 2:
            continue
        points = contour[:, 0, :].astype(np.float32) + np.array([x, y], np.float32)
        points /= np.array([w, h], np.float32)
        segments.append(np.clip(points, 0, 1))
    return segments


def directory_bytes(root, suffix=None):
    root = Path(root)
    if not root.is_dir():
        return 0
    return sum(
        p.stat().st_size
        for p in root.rglob("*")
        if p.is_file()
        and not p.is_symlink()
        and (suffix is None or p.suffix.lower() == suffix)
    )


def disk_space_plan(
    free_gib,
    experiment_gib=0.0,
    dataset_gib=0.0,
    weights_gib=0.0,
    numpy_cache_gib=0.0,
    dataset_complete=False,
):
    """Reserve peak simultaneous storage, rather than the sum of sequential stages."""
    count = train_records.file_name.nunique() + calibration_records.file_name.nunique()
    cal_count = calibration_records.file_name.nunique()
    report_count = report_records.file_name.nunique()
    test_count = sum(
        p.is_file() and p.suffix.lower() in {".jpg", ".jpeg"}
        for p in TEST_IMAGE_DIR.iterdir()
    )
    saved_candidates = (
        (NUM_EPOCHS + 2 if SAVE_EVERY_EPOCH else 2) if RUN_TRAINING else 1
    )
    pixels = IMAGE_SIZE**2
    # Probability uint8 + bit-packed single-view unions + salience + ZIP/JSON overhead.
    map_bytes = (
        pixels * (len(CONFIDENCE_CANDIDATES) + 1)
        + len(CONFIDENCE_CANDIDATES) * IMAGE_SIZE * ((IMAGE_SIZE + 7) // 8)
        + 64 * 1024
    )
    if PREDICTION_CACHE_POLICY == "all":
        peak_images = (
            cal_count * saved_candidates + report_count + test_count + NUM_WORKERS
        )
    else:
        # At calibration: incumbent winner and current candidate. At report/test:
        # winning calibration, preview maps, and at most four in-flight writes.
        calibration_peak = cal_count * min(2, saved_candidates)
        export_peak = (
            cal_count
            + min(report_count, PREDICTION_PREVIEW_COUNT)
            + min(test_count, PREDICTION_PREVIEW_COUNT)
        )
        peak_images = max(calibration_peak, export_peak) + NUM_WORKERS
    dataset_growth = (
        0.0 if dataset_complete else max(0.0, count * pixels * 1.1 / GIB - dataset_gib)
    )
    weights_growth = (
        max(0.0, max(2.0, saved_candidates * 0.20) - weights_gib)
        if RUN_TRAINING
        else 0.0
    )
    prediction_peak = peak_images * map_bytes / GIB
    cache_growth = max(0.0, count * pixels * 3 * 1.25 / GIB - numpy_cache_gib)
    budget = max(0.0, min(free_gib, DISK_LIMIT_GIB - experiment_gib))
    required = dataset_growth + weights_growth + prediction_peak + DISK_RESERVE_GIB
    return dict(
        policy=PREDICTION_CACHE_POLICY,
        budget_gib=budget,
        dataset_growth_gib=dataset_growth,
        checkpoint_growth_gib=weights_growth,
        prediction_peak_gib=prediction_peak,
        peak_cached_images=int(peak_images),
        reserve_gib=DISK_RESERVE_GIB,
        required_gib=required,
        image_cache_growth_gib=cache_growth,
        yolo_cache="disk" if budget >= required + cache_growth else False,
    )


def resource_preflight():
    available = psutil.virtual_memory().available / GIB
    try:
        limit = int(Path("/sys/fs/cgroup/memory.max").read_text())
        used = int(Path("/sys/fs/cgroup/memory.current").read_text())
        available = min(available, (limit - used) / GIB)
    except (OSError, ValueError):
        pass
    if available < RAM_RESERVE_GIB + 2:
        raise MemoryError(
            f"Only {available:.1f} GiB available; free memory before proceeding."
        )
    manifest = YOLO_DATA_DIR / "manifest.json"
    complete = (
        not REBUILD_YOLO_DATASET
        and manifest.is_file()
        and json.loads(manifest.read_text())["fingerprint"] == RUN_FINGERPRINT
    )
    numpy_gib = directory_bytes(YOLO_DATA_DIR, ".npy") / GIB
    plan = disk_space_plan(
        shutil.disk_usage(EXPERIMENT_DIR).free / GIB,
        experiment_gib=directory_bytes(EXPERIMENT_DIR) / GIB,
        dataset_gib=max(0.0, directory_bytes(YOLO_DATA_DIR) / GIB - numpy_gib),
        weights_gib=directory_bytes(RUNS_DIR, ".pt") / GIB,
        numpy_cache_gib=numpy_gib,
        dataset_complete=complete,
    )
    save_json(EXPERIMENT_DIR / "resource_plan.json", plan)
    if plan["budget_gib"] < plan["required_gib"]:
        raise OSError(
            f"Need {plan['required_gib']:.1f} GiB peak disk headroom, have {plan['budget_gib']:.1f}. "
            f"Prediction-cache policy: {PREDICTION_CACHE_POLICY}. See resource_plan.json."
        )
    print(
        f"Available RAM {available:.1f} GiB; disk headroom {plan['budget_gib']:.1f} GiB; "
        f"planned peak {plan['required_gib']:.1f} GiB including {DISK_RESERVE_GIB:.1f} GiB reserve; "
        f"prediction maps {plan['prediction_peak_gib']:.1f} GiB; YOLO image cache={plan['yolo_cache']}"
    )
    return plan["yolo_cache"]


def prepare_record(task):
    row, split_name = task
    image = read_gray(TRAIN_IMAGE_DIR / row["file_name"])
    union = merged_annotation_mask(coco, row["image_ids"])
    if union.shape != image.shape:
        raise ValueError("Annotation/image dimension mismatch.")
    stem = Path(row["file_name"]).stem
    image_path = YOLO_DATA_DIR / "images" / split_name / f"{stem}.png"
    label_path = YOLO_DATA_DIR / "labels" / split_name / f"{stem}.txt"
    processed = preprocess_grayscale(image)
    if not cv2.imwrite(str(image_path), processed, [cv2.IMWRITE_PNG_COMPRESSION, 3]):
        raise OSError(f"Could not save {image_path}")
    segments = mask_to_yolo_segments(union)
    lines = ["0 " + " ".join(f"{v:.7f}" for v in points.ravel()) for points in segments]
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    return dict(
        file_name=row["file_name"],
        split=split_name,
        annotators=len(row["image_ids"]),
        union_pixels=int(union.sum()),
        training_instances=len(segments),
    )


YOLO_CACHE_MODE = resource_preflight()
manifest_path = YOLO_DATA_DIR / "manifest.json"
if REBUILD_YOLO_DATASET and YOLO_DATA_DIR.exists():
    # Deletion is confined to this fingerprinted experiment's generated dataset.
    resolved = YOLO_DATA_DIR.resolve()
    if (
        resolved.parent != EXPERIMENT_DIR.resolve()
        or resolved.name != "yolo_filament_dataset"
    ):
        raise ValueError("Refusing to remove an unexpected dataset path.")
    shutil.rmtree(resolved)
if manifest_path.is_file():
    assert json.loads(manifest_path.read_text())["fingerprint"] == RUN_FINGERPRINT
    print("Reusing completed dataset:", YOLO_DATA_DIR)
else:
    if YOLO_DATA_DIR.exists() and any(YOLO_DATA_DIR.iterdir()):
        raise RuntimeError(
            "Incomplete dataset. Set REBUILD_YOLO_DATASET=True to rebuild this generated directory."
        )
    tasks = []
    for name, frame in [("train", train_records), ("val", calibration_records)]:
        for sub in ["images", "labels"]:
            (YOLO_DATA_DIR / sub / name).mkdir(parents=True, exist_ok=True)
        tasks.extend((row, name) for row in physical_rows(frame))
    stats = list(
        tqdm(bounded_map(prepare_record, tasks), total=len(tasks), desc="Union dataset")
    )
    pd.DataFrame(stats).to_csv(EXPERIMENT_DIR / "target_audit.csv", index=False)
    DATA_YAML.write_text(
        f"path: {json.dumps(YOLO_DATA_DIR.as_posix())}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n",
        encoding="utf-8",
    )
    save_json(manifest_path, {"fingerprint": RUN_FINGERPRINT, "samples": len(tasks)})
for name, frame in [("train", train_records), ("val", calibration_records)]:
    expected = {Path(n).stem for n in frame.file_name.unique()}
    for sub, ext in [("images", ".png"), ("labels", ".txt")]:
        actual = {p.stem for p in (YOLO_DATA_DIR / sub / name).glob("*" + ext)}
        assert actual == expected, f"Incomplete or stale {name} {sub}."
print(DATA_YAML.read_text())

In [ ]:
def pick_evenly(items, count):
    items = list(items)
    return (
        [
            items[int(i)]
            for i in np.linspace(0, len(items) - 1, min(count, len(items)), dtype=int)
        ]
        if items
        else []
    )


def show_gray(ax, image):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")


def overlay(ax, image, masks):
    # One RGB display array per panel. A separate RGBA artist per instance
    # would retain gigabytes for a crowded 2048 image.
    rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    for i, mask in enumerate(masks):
        color = np.array(plt.get_cmap("tab20")(i % 20)[:3]) * 255
        rgb[mask] = np.rint(rgb[mask] * 0.55 + color * 0.45).astype(np.uint8)
    ax.imshow(rgb, interpolation="nearest")
    ax.axis("off")


def finish_figure(fig, stage, filename):
    destination = PREVIEW_DIR / stage / f"{Path(filename).stem}.png"
    destination.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(destination, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


for row in pick_evenly(physical_rows(train_records), PREPROCESS_PREVIEW_COUNT):
    image = read_gray(TRAIN_IMAGE_DIR / row["file_name"])
    union = merged_annotation_mask(coco, row["image_ids"]).astype(bool)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, method in zip(axes.ravel()[:4], ["raw", "bilateral", "solar", "clahe"]):
        show_gray(ax, preprocess_grayscale(image, method))
        ax.set_title(method + (" (selected)" if method == PREPROCESSING_METHOD else ""))
    overlay(axes[1, 1], image, [union])
    axes[1, 1].set_title(f'Union of {len(row["image_ids"])} annotator records')
    target = np.zeros(image.shape, np.uint8)
    path = YOLO_DATA_DIR / "labels/train" / f'{Path(row["file_name"]).stem}.txt'
    for line in path.read_text().splitlines():
        points = np.array(line.split()[1:], np.float32).reshape(-1, 2) * [
            IMAGE_SIZE,
            IMAGE_SIZE,
        ]
        cv2.fillPoly(target, [np.rint(points).astype(np.int32)], 1)
    overlay(axes[1, 2], preprocess_grayscale(image), [target.astype(bool)])
    intersection = np.logical_and(target, union).sum()
    denominator = np.logical_or(target, union).sum()
    axes[1, 2].set_title(
        f"Exported YOLO target; union IoU={intersection/max(denominator,1):.4f}"
    )
    fig.suptitle(row["file_name"])
    finish_figure(fig, "preprocessing_targets", row["file_name"])

## 5. Train pretrained YOLO11s-seg; select best/last by PQ

DDP uses both GPUs with global batch two and two training-loader workers per rank. The pinned trainer normally creates additional validation workers; this small trainer subclass uses **zero validation workers** and one image per validation batch. Thus validation does not create another process pool beside the four training workers. It changes resource handling only, using the standard YOLO model, loss and optimizer. The pinned release serializes notebook-defined trainer classes for DDP.

Only calibration images enter YOLO validation. `best.pt` and `last.pt` compete later on calibration PQ. Epoch checkpoint selection is optional and disabled by default. Duplicate weight files are removed from the candidate list by SHA256. Report images remain unseen until selection is frozen.

Model definitions: [YOLO11 segmentation](https://github.com/ultralytics/ultralytics/blob/v8.4.171/ultralytics/cfg/models/11/yolo11-seg.yaml), [YOLO26 segmentation](https://github.com/ultralytics/ultralytics/blob/v8.4.171/ultralytics/cfg/models/26/yolo26-seg.yaml). The shared native decoder is checked against the YOLO11 output format; detection NMS runs through Ultralytics before mask decoding.

The pinned package may download `yolo26n.pt` for its one-time AMP numerical-precision check. The trained architecture is the YOLO11s-seg model printed in `model_description.json` and the training summary.


In [ ]:
class FourWorkerSegmentationTrainer(SegmentationTrainer):
    def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode="train"):
        if mode == "train":
            return super().get_dataloader(dataset_path, batch_size, rank, mode)
        previous = self.args.workers
        try:
            self.args.workers = 0
            return super().get_dataloader(dataset_path, 1, rank, mode)
        finally:
            self.args.workers = previous


training_start = time.perf_counter()
if RUN_TRAINING:
    model = YOLO(YOLO_MODEL_NAME)
    if model.task != "segment":
        raise ValueError(
            f"Expected an instance-segmentation model, got task={model.task}."
        )
    model_description = dict(
        pretrained=YOLO_MODEL_NAME,
        task=model.task,
        head=type(model.model.model[-1]).__name__,
        parameters=sum(p.numel() for p in model.model.parameters()),
    )
    save_json(EXPERIMENT_DIR / "model_description.json", model_description)
    print("Training model:", model_description)
    train_result = model.train(
        trainer=FourWorkerSegmentationTrainer,
        data=str(DATA_YAML),
        cache=YOLO_CACHE_MODE,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=False,
        **TRAIN_OPTIONS,
    )
    run_dir = Path(model.trainer.save_dir)
    checkpoint_candidates = [
        run_dir / "weights" / name for name in ["best.pt"]
    ]
    assert all(p.is_file() for p in checkpoint_candidates)
    if SAVE_EVERY_EPOCH:
        checkpoint_candidates += sorted((run_dir / "weights").glob("epoch*.pt"))
    unique_checkpoints = {}
    for candidate in checkpoint_candidates:
        unique_checkpoints.setdefault(file_sha256(candidate), candidate)
    checkpoint_candidates = list(unique_checkpoints.values())
    save_json(run_dir / "run_config.json", RUN_CONFIG)
    del train_result, model
else:
    if not EXISTING_CHECKPOINT:
        raise ValueError("Set EXISTING_CHECKPOINT when RUN_TRAINING=False.")
    checkpoint_candidates = [Path(EXISTING_CHECKPOINT).resolve()]
    run_dir = checkpoint_candidates[0].parent.parent
    previous = json.loads((run_dir / "run_config.json").read_text(encoding="utf-8"))
    if training_signature(previous) != training_signature(RUN_CONFIG):
        raise ValueError(
            "Checkpoint transform/targets/split/config differs from this run."
        )
training_seconds = time.perf_counter() - training_start
gc.collect()
if GPU_IDS:
    torch.cuda.empty_cache()
print("Checkpoint candidates:", checkpoint_candidates)
history_path = run_dir / "results.csv"
if history_path.is_file():
    history = pd.read_csv(history_path)
    history.columns = history.columns.str.strip()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for column in ["train/seg_loss", "val/seg_loss"]:
        if column in history:
            axes[0].plot(np.arange(1, len(history) + 1), history[column], label=column)
    for column in ["metrics/mAP50(M)", "metrics/mAP50-95(M)"]:
        if column in history:
            axes[1].plot(np.arange(1, len(history) + 1), history[column], label=column)
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.2)
        if ax.lines:
            ax.legend()
    axes[0].set_title("Mask loss")
    axes[1].set_title("Calibration mask mAP (not PQ)")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## Check Training History


In [ ]:
yolo_visual_groups = {
    "Overall Performance & Loss Tracking": ["results.png"],
    "Bounding Box Localization Metrics": [
        "BoxF1_curve.png",
        "BoxPR_curve.png",
        "BoxP_curve.png",
        "BoxR_curve.png",
    ],
    "Instance Segmentation Mask Metrics": [
        "MaskF1_curve.png",
        "MaskPR_curve.png",
        "MaskP_curve.png",
        "MaskR_curve.png",
    ],
    "Confusion Matrices": ["confusion_matrix.png", "confusion_matrix_normalized.png"],
    "Labels": ["labels.jpg"],
    "Training Dataset Batches & Anchors": [
        "train_batch0.jpg",
        "train_batch1.jpg",
        "train_batch2.jpg",
    ],
    "Validation Predictions vs Ground Truth": [
        "val_batch0_labels.jpg",
        "val_batch0_pred.jpg",
        "val_batch1_labels.jpg",
        "val_batch1_pred.jpg",
        "val_batch2_labels.jpg",
        "val_batch2_pred.jpg",
    ],
}


display_settings = {
    "Overall Performance & Loss Tracking": {
        "cols": 1,
        "col_width": 16,
        "row_height": 9,
    },
    "Bounding Box Localization Metrics": {
        "cols": 2,
        "col_width": 8,
        "row_height": 6,
    },
    "Instance Segmentation Mask Metrics": {
        "cols": 2,
        "col_width": 8,
        "row_height": 6,
    },
    "Confusion Matrices": {
        "cols": 1,
        "col_width": 10,
        "row_height": 8,
    },
    "Labels": {
        "cols": 1,
        "col_width": 12,
        "row_height": 8,
    },
    "Training Dataset Batches & Anchors": {
        "cols": 3,
        "col_width": 6,
        "row_height": 5,
    },
    "Validation Predictions vs Ground Truth": {
        "cols": 2,
        "col_width": 6,
        "row_height": 5,
    },
}


print(f"Loading generated artifact visuals from: {run_dir}\n" + "=" * 70)


for group_name, plot_files in yolo_visual_groups.items():
    valid_files = [f for f in plot_files if (run_dir / f).is_file()]

    if not valid_files:
        continue

    print(f"\n▶ Displaying Group: {group_name}")
    print("-" * 70)

    settings = display_settings[group_name]

    cols = settings["cols"]
    col_width = settings["col_width"]
    row_height = settings["row_height"]

    num_plots = len(valid_files)
    num_rows = (num_plots + cols - 1) // cols

    fig, axes = plt.subplots(
        num_rows,
        cols,
        figsize=(
            col_width * cols,
            row_height * num_rows,
        ),
    )

    axes = np.atleast_1d(axes).flatten()

    for idx, plot_name in enumerate(valid_files):
        img = cv2.imread(str(run_dir / plot_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        axes[idx].imshow(img)
        axes[idx].set_title(
            plot_name,
            fontsize=11,
            fontweight="bold",
            pad=8,
        )
        axes[idx].axis("off")

    for idx in range(num_plots, len(axes)):
        axes[idx].axis("off")

    fig.tight_layout()
    plt.show()

## 6. Shared dual-GPU inference and component extraction

The custom predictor retains **sigmoid mask probabilities**, using the same native logit upsampling and box crop as the pinned YOLO decoder, in chunks of four instances. Per detection-confidence threshold, instance probabilities are combined by maximum; these semantic foreground maps are averaged over inverse-transformed D4 views. Detection confidence selects instances; it is never treated as a pixel probability. Missing predictions contribute zero. The original view's binary union is stored separately for the baseline candidate.

One model replica owns each GPU. Four bounded CPU tasks share a queue of GPU lanes; each lane runs only one view at a time. This avoids sharing a YOLO predictor between concurrent calls. Cache metadata fingerprints weights, preprocessing, source file, confidence grid, TTA and decoder version. A cache hit performs no inference. Full-resolution maps are stored as compressed uint8 NPZ files; no full dataset of maps is held in RAM.

Hysteresis components must pass peak, mean probability and area checks. Morphology candidates can add only salience pixels near YOLO's weak foreground and above p > 0.15, with native 3/5-pixel closing. They cannot invent remote morphology-only instances. Every mode returns a disjoint component partition and uses the same code for calibration, report and submission.

With bounded cache retention, only winning calibration maps and report/test preview maps persist after a complete run. Each report/test map is deleted only after scoring or CSV-row encoding succeeds. To retune and re-export all report/test images later, rerun those inference stages; use `PREDICTION_CACHE_POLICY="all"` to retain all maps when space permits. Cleanup is confined to generated NPZ/JSON files inside this experiment's prediction-cache directories.


In [ ]:
def encode_mask(mask):
    return mask_utils.encode(np.asfortranarray(mask, dtype=np.uint8))


def json_rle(rle):
    return {
        "size": list(rle["size"]),
        "counts": (
            rle["counts"].decode("ascii")
            if isinstance(rle["counts"], bytes)
            else rle["counts"]
        ),
    }


def native_rle(rle):
    return {
        "size": list(rle["size"]),
        "counts": (
            rle["counts"].encode("ascii")
            if isinstance(rle["counts"], str)
            else rle["counts"]
        ),
    }


def decode_mask(rle):
    return mask_utils.decode(native_rle(rle)).astype(bool)


D4 = [
    (0, False),
    (0, True),
    (2, False),
    (2, True),
    (1, False),
    (1, True),
    (3, False),
    (3, True),
]


def d4_transform(array, k, flip):
    out = np.flip(array, axis=1) if flip else array
    return np.ascontiguousarray(np.rot90(out, k, axes=(0, 1)))


def d4_inverse_maps(array, k, flip):
    out = np.rot90(array, -k, axes=(-2, -1))
    return np.ascontiguousarray(np.flip(out, axis=-1) if flip else out)


class NativeProbabilityPredictor(SegmentationPredictor):
    def construct_result(self, pred, img, orig_img, img_path, proto):
        shape = orig_img.shape[:2]
        maps = torch.zeros(
            (len(CONFIDENCE_CANDIDATES), *shape),
            dtype=torch.float32,
            device=proto.device,
        )
        if len(pred):
            pred[:, :4] = ops.scale_boxes(img.shape[2:], pred[:, :4], orig_img.shape)
            channels, mh, mw = proto.shape
            # Sigmoid after native logit interpolation, then zero OUTSIDE each box.
            for start in range(0, len(pred), MASK_DECODE_CHUNK):
                detections = pred[start : start + MASK_DECODE_CHUNK]
                logits = (
                    detections[:, 6:].float() @ proto.float().view(channels, -1)
                ).view(-1, mh, mw)
                probs = ops.scale_masks(logits[None], shape)[0].sigmoid()
                probs = ops.crop_mask(probs, detections[:, :4])
                for index, threshold in enumerate(CONFIDENCE_CANDIDATES):
                    keep = detections[:, 4] >= threshold
                    if keep.any():
                        torch.maximum(maps[index], probs[keep].amax(0), out=maps[index])
        result = Results(
            orig_img, path=img_path, names=self.model.names, boxes=pred[:, :6]
        )
        result.single_binary = maps.gt(0.5).to(torch.uint8).cpu().numpy()
        result.probability_maps = (
            maps.mul(255).round().clamp_(0, 255).to(torch.uint8).cpu().numpy()
        )
        return result


class InferenceEngine:
    """One independently initialized YOLO/predictor per device; queue grants exclusive ownership."""

    def __init__(self, checkpoint):
        self.lanes = queue.Queue()
        self.models = []
        for name in INFERENCE_DEVICES:
            device = torch.device(
                name
            )  # Avoid changing CUDA_VISIBLE_DEVICES per worker.
            model = YOLO(str(checkpoint))
            predictor = NativeProbabilityPredictor(
                overrides=dict(
                    task="segment",
                    mode="predict",
                    imgsz=IMAGE_SIZE,
                    batch=1,
                    device=device,
                    quantize=16 if device.type == "cuda" else None,
                    conf=min(CONFIDENCE_CANDIDATES),
                    max_det=MAX_DETECTIONS,
                    retina_masks=True,
                    verbose=False,
                    save=False,
                )
            )
            # Initialize/fuse serially: no first-call global setup race across GPU threads.
            predictor.setup_model(model=model.model, verbose=False)
            model.predictor = predictor
            self.models.append(model)
            self.lanes.put((model, device))

    def predict_maps(self, processed):
        model, device = self.lanes.get()
        try:
            total = np.zeros((len(CONFIDENCE_CANDIDATES), *processed.shape), np.uint16)
            single_binary = None
            for view_index, (k, flip) in enumerate(D4[:TTA_VIEWS]):
                view = d4_transform(processed, k, flip)
                result = model.predict(
                    source=cv2.cvtColor(view, cv2.COLOR_GRAY2BGR),
                    imgsz=IMAGE_SIZE,
                    conf=min(CONFIDENCE_CANDIDATES),
                    iou=PREDICT_NMS_IOU,
                    retina_masks=True,
                    max_det=MAX_DETECTIONS,
                    device=device,
                    quantize=16 if device.type == "cuda" else None,
                    batch=1,
                    nms=None,
                    augment=False,
                    save=False,
                    verbose=False,
                )[0]
                total += d4_inverse_maps(result.probability_maps, k, flip)
                if view_index == 0:
                    single_binary = result.single_binary
                del result
            probability = np.rint(total.astype(np.float32) / TTA_VIEWS).astype(np.uint8)
            return probability, single_binary
        finally:
            self.lanes.put((model, device))

    def close(self):
        while not self.lanes.empty():
            self.lanes.get_nowait()
        self.models.clear()
        gc.collect()
        for device in GPU_IDS:
            with torch.cuda.device(device):
                torch.cuda.empty_cache()

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()


def current_inference_config():
    return dict(
        tta_views=TTA_VIEWS,
        confidences=list(CONFIDENCE_CANDIDATES),
        decoder="native_sigmoid_chunked_v1",
        image_size=IMAGE_SIZE,
        preprocessing_method=PREPROCESSING_METHOD,
        preprocessing=dict(PREPROCESSING_CONFIG),
        predict_nms_iou=PREDICT_NMS_IOU,
        max_det=MAX_DETECTIONS,
    )


def inference_cache_dir(checkpoint):
    signature = dict(
        weights=file_sha256(checkpoint),
        run=RUN_FINGERPRINT,
        inference=current_inference_config(),
    )
    root = EXPERIMENT_DIR / "prediction_cache" / fingerprint(signature)[:16]
    root.mkdir(parents=True, exist_ok=True)
    return root


def checked_prediction_cache_dir(cache_dir):
    path = Path(cache_dir)
    expected = EXPERIMENT_DIR / "prediction_cache"
    resolved = path.resolve()
    if (
        path.is_symlink()
        or expected.is_symlink()
        or resolved.parent != expected.resolve()
        or re.fullmatch(r"[0-9a-f]{16}", resolved.name) is None
    ):
        raise ValueError(
            f"Refusing to prune outside this experiment's generated prediction cache: {path}"
        )
    return resolved


def delete_prediction_source(cache_dir, stem):
    root = checked_prediction_cache_dir(cache_dir)
    if Path(stem).name != stem or stem in {"", ".", ".."}:
        raise ValueError("Invalid prediction-cache stem.")
    for suffix in [".npz", ".json", ".npz.tmp", ".json.tmp"]:
        path = root / (stem + suffix)
        if path.is_symlink():
            raise ValueError(f"Refusing to prune a linked cache file: {path}")
        path.unlink(missing_ok=True)


def prune_prediction_cache(cache_dir, keep_stems=()):
    root = checked_prediction_cache_dir(cache_dir)
    keep = set(keep_stems)
    removed = 0
    for path in list(root.iterdir()):
        # Only our flat per-image NPZ/JSON artifacts; never weights or datasets.
        suffix = next(
            (
                s
                for s in [".npz.tmp", ".json.tmp", ".npz", ".json"]
                if path.name.endswith(s)
            ),
            None,
        )
        if suffix is None or not path.is_file():
            continue
        if path.name[: -len(suffix)] not in keep:
            if path.is_symlink():
                raise ValueError(f"Refusing to prune a linked cache file: {path}")
            removed += path.stat().st_size
            path.unlink()
    return removed


def cached_predict(engine, path, cache_dir):
    path = Path(path)
    destination = cache_dir / f"{path.stem}.json"
    map_path = cache_dir / f"{path.stem}.npz"
    stat = path.stat()
    source_signature = [str(path.resolve()), stat.st_size, stat.st_mtime_ns]
    if destination.is_file() and map_path.is_file():
        cached = json.loads(destination.read_text(encoding="utf-8"))
        if cached["source"] == source_signature and cached.get(
            "inference_signature"
        ) == fingerprint(current_inference_config()):
            cached["map_path"] = str(map_path)
            return cached
    if engine is None:
        raise RuntimeError(
            f"Cache missing/stale for {path.name}; rerun its inference stage."
        )
    image = read_gray(path)
    probability, single_binary = engine.predict_maps(preprocess_grayscale(image))
    disk = estimate_disk(image)
    salience = morphology_salience_u8(image, disk)
    temporary = map_path.with_suffix(".npz.tmp")
    with temporary.open("wb") as stream:
        np.savez_compressed(
            stream,
            probability=probability,
            single_binary=np.packbits(single_binary, axis=-1),
            salience=salience,
        )
    os.replace(temporary, map_path)
    cached = dict(
        source=source_signature,
        file_name=path.name,
        disk=disk,
        map_path=str(map_path),
        confidences=CONFIDENCE_CANDIDATES,
        tta_views=TTA_VIEWS,
        inference_signature=fingerprint(current_inference_config()),
    )
    save_json(destination, cached)  # Publish metadata only after the maps are complete.
    return cached


def load_maps(cached):
    with np.load(cached["map_path"], allow_pickle=False) as data:
        maps = {key: data[key] for key in ["probability", "single_binary", "salience"]}
    maps["single_binary"] = np.unpackbits(
        maps["single_binary"], axis=-1, count=maps["probability"].shape[-1]
    )
    return maps


def component_base_key(cfg):
    return (
        cfg["confidence"],
        cfg["mode"],
        cfg["clip_to_disk"],
        cfg.get("t_lo", 0.5),
        cfg.get("bridge_px", 0),
    )


def component_candidates(cached, maps, cfg):
    index = cached["confidences"].index(cfg["confidence"])
    p = maps["probability"][index]
    if cfg["mode"] == "union_cc":
        binary = maps["single_binary"][index].copy()
    else:
        binary = (p > int(round(cfg["t_lo"] * 255))).astype(np.uint8)
        if cfg["mode"] == "hysteresis_morph" and cached["disk"] is not None:
            width = cfg["bridge_px"]
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (width, width))
            near = cv2.dilate(binary, kernel).astype(bool)
            permitted = p > int(round(0.15 * 255))
            extension = near & permitted & (maps["salience"] >= 48)  # z >= 1.5
            binary |= extension.astype(np.uint8)
            binary = cv2.morphologyEx(
                binary, cv2.MORPH_CLOSE, kernel
            ) & permitted.astype(np.uint8)
    if cfg["clip_to_disk"]:
        binary &= disk_support(binary.shape, cached["disk"]).astype(np.uint8)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    output = []
    for label in range(1, count):
        x, y, w, h, area = (int(v) for v in stats[label])
        roi = labels[y : y + h, x : x + w] == label
        values = p[y : y + h, x : x + w][roi]
        full = np.zeros(binary.shape, np.uint8)
        full[y : y + h, x : x + w] = roi
        output.append(
            dict(
                rle=encode_mask(full),
                area=area,
                meanp=float(values.mean()) / 255,
                peak=int(values.max()),
                x=x,
                y=y,
            )
        )
    return output


def keep_components(candidates, cfg):
    kept = [c for c in candidates if c["area"] >= max(1, cfg["min_area"])]
    if cfg["mode"] != "union_cc":
        kept = [
            c
            for c in kept
            if c["peak"] > int(round(cfg["t_hi"] * 255))
            and c["meanp"] >= cfg["min_meanp"]
        ]
    return [c["rle"] for c in sorted(kept, key=lambda c: (-c["area"], c["y"], c["x"]))]


def canonical_postprocessing_config(config):
    cfg = dict(config)
    if cfg.get("mode") == "instances":
        cfg["mode"] = "union_cc"  # Compatibility with old disjoint-export repair.
    if cfg["mode"] not in {"union_cc", "hysteresis", "hysteresis_morph"}:
        raise ValueError(f"Unknown postprocessing mode: {cfg['mode']}")
    return cfg


def final_prediction(cached, config):
    cfg = canonical_postprocessing_config(config)
    return keep_components(component_candidates(cached, load_maps(cached), cfg), cfg)


POSTPROCESS_GRID = []
for confidence in CONFIDENCE_CANDIDATES:
    for clip in CLIP_TO_DISK_CANDIDATES:
        base = dict(confidence=confidence, clip_to_disk=clip)
        POSTPROCESS_GRID.extend(
            dict(**base, mode="union_cc", min_area=area) for area in AREA_CANDIDATES
        )
        POSTPROCESS_GRID.extend(
            dict(**base, mode="hysteresis", **rule) for rule in KEEP_RULES
        )
        POSTPROCESS_GRID.extend(
            dict(**base, mode="hysteresis_morph", **rule) for rule in MORPHOLOGY_RULES
        )
print(
    f"Calibration candidates: {len(POSTPROCESS_GRID)} per checkpoint; GPUs: {INFERENCE_DEVICES}; CPU pool: {NUM_WORKERS}"
)

In [ ]:
def target_rles(image_id):
    rles = [coco.annToRLE(a) for a in coco.loadAnns(coco.getAnnIds(imgIds=[image_id]))]
    return [r for r in rles if int(mask_utils.area(r)) > 0]


def score_rles(predictions, targets):
    n_pred, n_gt = len(predictions), len(targets)
    overlaps = (
        mask_utils.iou(predictions, targets, [0] * n_gt).T
        if n_pred and n_gt
        else np.zeros((n_gt, n_pred), np.float64)
    )
    hits = overlaps > PQ_MATCH_IOU
    # Exactly the organizers' published self-evaluation logic, including overlap cases.
    tp = int(hits.sum())
    fp = int((hits.sum(axis=0) == 0).sum())
    fn = int((hits.sum(axis=1) == 0).sum())
    iou_sum = float(overlaps[hits].sum())
    den = tp + 0.5 * (fp + fn)
    pred_union = mask_utils.merge(predictions) if predictions else None
    gt_union = mask_utils.merge(targets) if targets else None
    pred_area = int(mask_utils.area(pred_union)) if pred_union is not None else 0
    gt_area = int(mask_utils.area(gt_union)) if gt_union is not None else 0
    intersection = (
        int(mask_utils.area(mask_utils.merge([pred_union, gt_union], intersect=True)))
        if pred_area and gt_area
        else 0
    )
    return dict(
        tp=tp,
        fp=fp,
        fn=fn,
        matched_iou_sum=iou_sum,
        pq=iou_sum / den if den else 0.0,
        one_to_many=int((hits.sum(axis=1) > 1).sum()),
        many_to_one=int((hits.sum(axis=0) > 1).sum()),
        n_pred=n_pred,
        n_gt=n_gt,
        pixel_dice=(
            2 * intersection / (pred_area + gt_area) if pred_area + gt_area else 1.0
        ),
        pixel_iou=(
            intersection / (pred_area + gt_area - intersection)
            if pred_area + gt_area
            else 1.0
        ),
    )


def pooled_metrics(rows):
    frame = pd.DataFrame(rows)
    tp, fp, fn = (int(frame[c].sum()) for c in ["tp", "fp", "fn"])
    iou_sum = float(frame.matched_iou_sum.sum())
    denominator = tp + 0.5 * (fp + fn)
    return dict(
        PQ=iou_sum / denominator if denominator else 0.0,
        SQ=iou_sum / tp if tp else 0.0,
        RQ=tp / denominator if denominator else 0.0,
        TP=tp,
        FP=fp,
        FN=fn,
        mean_Dice=float(frame.pixel_dice.mean()),
        mean_IoU=float(frame.pixel_iou.mean()),
        annotator_records=len(frame),
        one_to_many=int(frame.one_to_many.sum()),
        many_to_one=int(frame.many_to_one.sum()),
    )


def score_one_source(cached, image_ids, configs):
    targets = {image_id: target_rles(image_id) for image_id in image_ids}
    maps = load_maps(cached)
    output, bases = [], {}
    for config_idx, cfg in enumerate(configs):
        key = component_base_key(cfg)
        if key not in bases:
            bases[key] = component_candidates(cached, maps, cfg)
        prediction = keep_components(bases[key], cfg)
        for image_id, gt in targets.items():
            output.append(
                dict(
                    config_idx=config_idx,
                    image_id=image_id,
                    file_name=cached["file_name"],
                    **score_rles(prediction, gt)
                )
            )
    return output


def evaluate_partition(
    engine, frame, configs, cache_dir, description, retain_cache_stems=None
):
    physical = physical_rows(frame)

    def task(row):
        cached = cached_predict(engine, TRAIN_IMAGE_DIR / row["file_name"], cache_dir)
        scores = score_one_source(cached, row["image_ids"], configs)
        if (
            retain_cache_stems is not None
            and Path(row["file_name"]).stem not in retain_cache_stems
        ):
            delete_prediction_source(cache_dir, Path(row["file_name"]).stem)
        return scores

    rows = []
    for result in tqdm(
        bounded_map(task, physical), total=len(physical), desc=description
    ):
        rows.extend(result)
    per_record = pd.DataFrame(rows)
    summaries = [
        dict(
            config_idx=i,
            **cfg,
            **pooled_metrics(per_record[per_record.config_idx == i].to_dict("records"))
        )
        for i, cfg in enumerate(configs)
    ]
    return per_record, pd.DataFrame(summaries)

## 7. Select on calibration, then freeze and report

The selected score on calibration is optimistic because it is used for selection. **Report PQ** is the separate estimate. Use these report scores sparingly when comparing bilateral/solar/raw/CLAHE; repeated tuning against the report also turns it into a selection set. Large leaderboard gains are not guaranteed. Per-annotator counts are pooled, rather than averaging per-image PQ.

Checkpoint candidates run sequentially. Under the default bounded policy, calibration keeps the current winner's maps and the candidate being evaluated, then prunes the losing cache. Strictly higher PQ replaces the incumbent, preserving the same stable tie rule as the summary table. Scores for every candidate remain in CSV and weight files are retained. Report images are scored in a four-task stream; only the selected preview maps are kept.


In [ ]:
validation_start = time.perf_counter()
all_summaries = []
best_cache_dir, best_cache_pq = None, -float("inf")
cache_retention_log = []
for checkpoint in checkpoint_candidates:
    cache_dir = inference_cache_dir(checkpoint)
    with InferenceEngine(checkpoint) as engine:
        per_record, summary = evaluate_partition(
            engine,
            calibration_records,
            POSTPROCESS_GRID,
            cache_dir,
            f"Calibration {checkpoint.name}",
        )
    summary["checkpoint"] = str(checkpoint.resolve())
    summary["weights_sha256"] = file_sha256(checkpoint)
    all_summaries.append(summary)
    per_record.to_csv(
        EXPERIMENT_DIR / f"calibration_{checkpoint.stem}_per_record.csv", index=False
    )
    candidate_pq = float(summary.PQ.max())
    removed_bytes = 0
    if PREDICTION_CACHE_POLICY == "bounded":
        if candidate_pq > best_cache_pq:
            if best_cache_dir is not None and best_cache_dir != cache_dir:
                removed_bytes = prune_prediction_cache(best_cache_dir)
            best_cache_dir, best_cache_pq = cache_dir, candidate_pq
        elif best_cache_dir != cache_dir:
            removed_bytes = prune_prediction_cache(cache_dir)
    cache_retention_log.append(
        dict(
            checkpoint=str(checkpoint),
            PQ=candidate_pq,
            removed_cache_bytes=removed_bytes,
        )
    )
    save_json(EXPERIMENT_DIR / "cache_retention_log.json", cache_retention_log)
    gc.collect()
    if GPU_IDS:
        torch.cuda.empty_cache()
calibration_summary = pd.concat(all_summaries, ignore_index=True)
calibration_summary = calibration_summary.sort_values(
    "PQ", ascending=False, kind="stable"
).reset_index(drop=True)
calibration_summary.to_csv(EXPERIMENT_DIR / "calibration_grid.csv", index=False)
winner = calibration_summary.iloc[0]
SELECTED_CONFIG = canonical_postprocessing_config(
    POSTPROCESS_GRID[int(winner.config_idx)]
)
SELECTED_CHECKPOINT = Path(winner.checkpoint)
SELECTED_CACHE_DIR = inference_cache_dir(SELECTED_CHECKPOINT)
FROZEN_SELECTION = dict(
    model=YOLO_MODEL_NAME,
    checkpoint=str(SELECTED_CHECKPOINT),
    weights_sha256=winner.weights_sha256,
    postprocessing=SELECTED_CONFIG,
    inference=current_inference_config(),
    mask_postprocessing_version="solar_d4_hysteresis_nonoverlap_v3",
    calibration_PQ=float(winner.PQ),
    run_fingerprint=RUN_FINGERPRINT,
)
save_json(EXPERIMENT_DIR / "selected_pipeline.json", FROZEN_SELECTION)
display(calibration_summary.head(12))
mode_comparison = (
    calibration_summary.sort_values("PQ", ascending=False)
    .groupby("mode", sort=False)
    .head(1)
)
mode_comparison.to_csv(EXPERIMENT_DIR / "calibration_mode_comparison.csv", index=False)
display(
    mode_comparison[["checkpoint", "mode", "confidence", "min_area", "PQ", "FP", "FN"]]
)
print("Frozen selection:", FROZEN_SELECTION)

report_preview_stems = {
    Path(row["file_name"]).stem
    for row in pick_evenly(physical_rows(report_records), PREDICTION_PREVIEW_COUNT)
}
with InferenceEngine(SELECTED_CHECKPOINT) as engine:
    report_per_record, report_summary = evaluate_partition(
        engine,
        report_records,
        [SELECTED_CONFIG],
        SELECTED_CACHE_DIR,
        "Held-out report",
        retain_cache_stems=(
            report_preview_stems if PREDICTION_CACHE_POLICY == "bounded" else None
        ),
    )
report_per_record.to_csv(EXPERIMENT_DIR / "report_per_annotator.csv", index=False)
report_summary.to_csv(EXPERIMENT_DIR / "report_metrics.csv", index=False)
validation_seconds = time.perf_counter() - validation_start
display(report_summary)
save_json(
    EXPERIMENT_DIR / "timings.json",
    dict(training_seconds=training_seconds, validation_seconds=validation_seconds),
)
fig, ax = plt.subplots(figsize=(8, 4))
for (ckpt, mode, clip), frame in calibration_summary.groupby(
    ["checkpoint", "mode", "clip_to_disk"]
):
    best_per_conf = frame.groupby("confidence").PQ.max()
    ax.plot(
        best_per_conf.index,
        best_per_conf.values,
        marker=".",
        label=f"{Path(ckpt).name}, {mode}, clip={clip}",
    )
ax.set(
    xlabel="Confidence",
    ylabel="Calibration PQ",
    title="Best keep rule per curve; calibration selection scores",
)
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
plt.close(fig)

## 8. Ten validation prediction previews

Evenly spaced distinct report images, including empty predictions. The diagnostic overlay shows the single-view union; the middle bottom panel shows the selected final masks. Peak/mean filters and TTA can change both count and boundaries. Metrics score every annotator separately; one annotator is displayed for readability. These cells read already-computed caches.


In [ ]:
def prediction_preview(path, cached, final_rles, stage, gt_rles=None):
    image = read_gray(path)
    processed = preprocess_grayscale(image)
    maps = load_maps(cached)
    raw = [maps["single_binary"][0].astype(bool)]
    final = [decode_mask(r) for r in final_rles]
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.ravel()
    show_gray(axes[0], image)
    axes[0].set_title("Original 2048 image")
    show_gray(axes[1], processed)
    axes[1].set_title(f"Model input: {PREPROCESSING_METHOD}")
    overlay(axes[2], image, raw)
    axes[2].set_title(f"Single-view union (conf ≥ {min(CONFIDENCE_CANDIDATES):.2f})")
    if gt_rles is not None:
        overlay(axes[3], image, [decode_mask(r) for r in gt_rles])
        axes[3].set_title("GT: one annotator (display only)")
    else:
        merged = np.zeros(image.shape, np.uint8)
        for mask in final:
            merged |= mask.astype(np.uint8)
        axes[3].imshow(merged, cmap="gray", vmin=0, vmax=1)
        axes[3].axis("off")
        axes[3].set_title("Final mask decoded from submission CSV")
    overlay(axes[4], image, final)
    axes[4].set_title(f"Final predicted instances: {len(final)}")
    if final_rles:
        x, y, w, h = mask_utils.toBbox(native_rle(final_rles[0])).astype(int)
        pad = max(40, int(max(w, h) * 0.20))
        x0, x1 = max(0, x - pad), min(IMAGE_SIZE, x + w + pad)
        y0, y1 = max(0, y - pad), min(IMAGE_SIZE, y + h + pad)
        overlay(axes[5], image[y0:y1, x0:x1], [m[y0:y1, x0:x1] for m in final])
        axes[5].set_title("Largest instance: native-pixel detail")
    else:
        show_gray(axes[5], image)
        axes[5].set_title("No predicted filaments")
    fig.suptitle(f"{Path(path).name} | {SELECTED_CONFIG}", fontsize=11)
    finish_figure(fig, stage, Path(path).name)


for row in pick_evenly(physical_rows(report_records), PREDICTION_PREVIEW_COUNT):
    path = TRAIN_IMAGE_DIR / row["file_name"]
    cached = cached_predict(None, path, SELECTED_CACHE_DIR)
    prediction_preview(
        path,
        cached,
        final_prediction(cached, SELECTED_CONFIG),
        "validation_predictions",
        target_rles(row["image_ids"][0]),
    )

## 9. Competition submission — original 0.36 CSV contract

The selected checkpoint, input transform, TTA and extraction rule are frozen before test inference. Both GPUs process test images through the same bounded four-worker path used for validation. CSV rows are written in sorted image order, with per-image IDs starting at one and masks sorted by area/top/left. The exact original `mask_to_rle`, header `filament_id,segmentation_rle`, default `csv.writer`, and native 2048 mask shape are retained.

Zero-prediction images produce no placeholder row but are recorded in the inference log. A temporary CSV is fully decoded, round-tripped, checked for unique IDs and overlapping masks, then atomically renamed. The audit verifies the output format, not competition performance.


In [ ]:
def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return encoded["counts"].decode("utf-8")


def get_test_image_paths():
    paths = [
        path
        for path in TEST_IMAGE_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg"}
    ]
    return sorted(paths, key=lambda path: path.name)


def test_image_paths():
    return get_test_image_paths()


def submission_rows_for_source(path_stem, rles):
    # The masks come from the same selected postprocessing used in validation.
    # Encode each native mask with the exact original encoder and ID loop.
    components = (decode_mask(rle) for rle in rles)
    rows = []
    for filament_counter, component in enumerate(components, start=1):
        if component.shape != (2048, 2048):
            raise ValueError("Submission masks must be exactly 2048 x 2048.")
        rows.append((f"{path_stem}_{filament_counter}", mask_to_rle(component)))
    return rows


def create_submission_parallel(output_path=SUBMISSION_PATH):
    # Accept both strings and Path objects, as open() did in the original.
    output_path = Path(output_path)
    if FROZEN_SELECTION.get("inference") != current_inference_config():
        raise ValueError(
            "Inference settings changed after selection; rerun calibration before export."
        )
    paths = get_test_image_paths()
    if not paths:
        raise FileNotFoundError(f"No .jpg/.jpeg images found in {TEST_IMAGE_DIR}")
    if len({path.stem for path in paths}) != len(paths):
        raise ValueError("Test images must have unique filename stems.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    processed = {}
    seen_filament_ids = set()
    preview_stems = {p.stem for p in pick_evenly(paths, PREDICTION_PREVIEW_COUNT)}
    start_time = time.perf_counter()
    export_config = canonical_postprocessing_config(SELECTED_CONFIG)
    export_selection = dict(FROZEN_SELECTION)
    export_selection["postprocessing"] = export_config
    export_selection["mask_postprocessing_version"] = (
        "solar_d4_hysteresis_nonoverlap_v3"
    )
    if export_config != SELECTED_CONFIG:
        # A previous score for overlapping instances does not describe this output.
        export_selection.pop("calibration_PQ", None)
        export_selection["recalibration_needed"] = True
        print(
            "Legacy instance mode replaced by disjoint union/components. Recalibrate for updated PQ.",
            flush=True,
        )

    with open(temporary, mode="w", newline="", encoding="utf-8") as f:
        # Exactly the original writer: default dialect, quoting and CRLF ending.
        writer = csv.writer(f)
        writer.writerow(["filament_id", "segmentation_rle"])

        def task(path):
            cached = cached_predict(engine, path, SELECTED_CACHE_DIR)
            rows = submission_rows_for_source(
                path.stem, final_prediction(cached, export_config)
            )
            if PREDICTION_CACHE_POLICY == "bounded" and path.stem not in preview_stems:
                delete_prediction_source(SELECTED_CACHE_DIR, path.stem)
            return path.stem, rows

        with InferenceEngine(SELECTED_CHECKPOINT) as engine:
            results = bounded_map(task, paths)
            for stem, worker_rows in tqdm(
                results, total=len(paths), desc="Submission, both GPUs"
            ):
                for fid, rle in worker_rows:
                    if fid in seen_filament_ids:
                        raise ValueError(f"Duplicate filament ID detected: {fid}")
                    seen_filament_ids.add(fid)
                    writer.writerow([fid, rle])
                processed[stem] = len(worker_rows)

    assert set(processed) == {path.stem for path in paths}
    audit = audit_submission(temporary, processed)
    os.replace(temporary, output_path)
    save_json(EXPERIMENT_DIR / "test_inference_log.json", processed)
    audit.update(
        sha256=file_sha256(output_path),
        inference_seconds=time.perf_counter() - start_time,
        csv=str(output_path),
        pipeline=export_selection,
        submission_format="original_0_36",
    )
    save_json(EXPERIMENT_DIR / "submission_audit.json", audit)
    print(
        f"\n[SUCCESS] Submission complete!\nProcessed: {len(paths):,} images"
        f"\nFilament rows written: {audit['submitted_filaments']:,}\nSaved: {output_path}",
        flush=True,
    )
    return audit


def create_submission(output_path=None):
    # Compatibility with the previous call name; serialization stays original.
    return create_submission_parallel(
        SUBMISSION_PATH if output_path is None else output_path
    )


def audit_submission(path, processed):
    seen = set()
    counts = {stem: 0 for stem in processed}
    # Compact accumulated RLE per image handles interleaved image rows as well.
    occupied_rles = {}
    with open(path, newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["filament_id", "segmentation_rle"]
        for row in reader:
            assert (
                set(row) == {"filament_id", "segmentation_rle"}
                and None not in row.values()
            )
            filament_id = row["filament_id"]
            stem, tail = filament_id.rsplit("_", 1)
            assert (
                stem in processed
                and tail.isdigit()
                and int(tail) > 0
                and filament_id not in seen
            )
            seen.add(filament_id)
            assert row["segmentation_rle"], "Empty RLE counts."
            rle = native_rle({"size": [2048, 2048], "counts": row["segmentation_rle"]})
            mask = decode_mask(rle)
            assert mask.shape == (2048, 2048) and mask.any()
            assert (
                mask_to_rle(mask) == row["segmentation_rle"]
            ), "Original-encoder RLE roundtrip failed."
            if stem in occupied_rles:
                intersection = mask_utils.merge(
                    [occupied_rles[stem], rle], intersect=True
                )
                overlap_pixels = int(mask_utils.area(intersection))
                if overlap_pixels:
                    raise ValueError(
                        f"Invalid submission: {filament_id} overlaps an earlier mask for {stem} "
                        f"by {overlap_pixels:,} pixels. The final CSV was not published."
                    )
                occupied_rles[stem] = mask_utils.merge([occupied_rles[stem], rle])
            else:
                occupied_rles[stem] = rle
            counts[stem] += 1
    assert counts == processed, "CSV rows differ from the complete test inference log."
    return dict(
        test_images_processed=len(processed),
        zero_prediction_images=sum(v == 0 for v in processed.values()),
        submitted_filaments=len(seen),
        decoded_every_rle=True,
        exact_columns=True,
        unique_ids=True,
        non_overlapping_masks=True,
        overlap_pixels=0,
    )


if CREATE_SUBMISSION:
    submission_audit = create_submission_parallel()
    print(json.dumps(submission_audit, indent=2))
else:
    print("CREATE_SUBMISSION=False: submission and test galleries skipped.")

## 10. Ten test prediction previews — masks actually written to CSV

Evenly spaced test files are selected without labels or confidence-based cherry-picking. The final masks are decoded from the finished CSV, so previews verify the submitted content. No second model inference is required. Each figure shows the original, processed input, returned predictions, submitted mask/overlay and a detail crop. PNGs are saved under `previews/test_predictions/`.


In [ ]:
if CREATE_SUBMISSION:
    preview_paths = pick_evenly(test_image_paths(), PREDICTION_PREVIEW_COUNT)
    preview_rles = {p.stem: [] for p in preview_paths}
    with Path(SUBMISSION_PATH).open(newline="", encoding="utf-8") as stream:
        for row in csv.DictReader(stream):
            stem = row["filament_id"].rsplit("_", 1)[0]
            if stem in preview_rles:
                preview_rles[stem].append(
                    {"size": [2048, 2048], "counts": row["segmentation_rle"]}
                )
    for path in preview_paths:
        cached = cached_predict(None, path, SELECTED_CACHE_DIR)
        prediction_preview(path, cached, preview_rles[path.stem], "test_predictions")
    print(
        f"Displayed and saved {len(preview_paths)} distinct test prediction previews."
    )

## Outputs and interpretation

- `solar_filament_yolo11s_submission.csv` in the working directory: **the single file to upload**, with the original 0.36 notebook's CSV contract.
- Fingerprinted experiment directory: `run_config.json`, grouped split CSVs, target audit, best/last checkpoints (epoch checkpoints only if enabled), `calibration_grid.csv`, `calibration_mode_comparison.csv`, and frozen `selected_pipeline.json`.
- `report_metrics.csv` and `report_per_annotator.csv`: separate held-out PQ, SQ, RQ, Dice/IoU and error counts. The calibration winner's score is optimistic because it selected the rule.
- Compressed `prediction_cache/`: per-confidence probabilities, first-view binary unions and salience. The default bounded policy retains winning calibration maps and report/test preview maps. Other report/test maps require inference again for later retuning; `"all"` retains everything when disk permits. Losing-checkpoint scores and all checkpoint weights remain available.
- `resource_plan.json` and `cache_retention_log.json`: peak disk estimates and generated-cache cleanup records.
- `test_inference_log.json`, `submission_audit.json`, and ten validation/test previews: coverage, exact RLE verification and non-overlap checks.

The selected run uses YOLO11s-seg with bilateral filtering, the configured epoch budget and D4 averaging. `model_description.json` records the actual loaded model head and parameter count. The model name is included in run fingerprints, so YOLO26 checkpoints and predictions cannot silently be reused. Calibration can retain the single-view union rule if it wins. A second clean run with `PREPROCESSING_METHOD="raw"` isolates the input-transform effect; setting `TTA_VIEWS=1` tests the averaging contribution. Retrain only when training inputs/settings change. Repeated experiments on the report split eventually make it another tuning set, so keep comparisons limited.

The observed YOLO26 plateau does not establish a 0.39 ceiling. No higher public score is asserted for YOLO11 by this edit. Run full training/calibration/report inference and submit to measure whether it improves the user-reported 0.32 preprocessing result or 0.36 baseline. The other three supplied notebooks remain unchanged; this notebook keeps its existing filename for continuity. The optional standalone CSV repair cell still works independently at the top.
